# 01 — Khám phá Corpus Wikipedia tiếng Việt & Kiểm định Tokenizer (`CustomBPE`)

Notebook tổng hợp toàn bộ quy trình **Tuần 2 (Thành viên 1)**:
- **(a)** Thống kê phân phối độ dài corpus sau làm sạch, lọc chất lượng và khử trùng lặp (`split_stats.json`).
- **(b)** Kiểm thử round-trip `CustomBPE` trên các ca biên tiếng Việt (NFC, NFD, emoji, ký tự đặc biệt, literal `<|endoftext|>`).
- **(c)** Bảng so sánh các cỡ từ vựng `[4096, 8192, 16384]` (Compression Ratio, Fertility, tốc độ) đối chiếu với HuggingFace `tokenizers`.
- **(d)** Chi phí tham số Embedding ($V \times d_{\text{model}}$ với weight tying) và lý do chốt `vocab_size = 8192`.
- **(e)** Kiểm định chất lượng nội bộ vocab (`inspect_tokenizer_vocab`) và phân tích tần suất / token chết trên `.bin` (`analyze_token_usage`).
- **(f)** Kiểm tra hợp đồng `meta.json` và đọc giải mã mẫu thực tế từ `test.bin`.

In [ ]:
from __future__ import annotations

import json
import sys
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

# Chuyển đổi giữa "colab" (dữ liệu thật trên Google Drive) và "fixtures" (chạy nhanh local)
MODE = "colab"  # Đổi thành "fixtures" nếu muốn chạy nhanh trên máy local

# Tự động xác định thư mục gốc repo (hỗ trợ chạy từ root hoặc từ notebooks/)
cwd = Path.cwd().resolve()
REPO_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.bpe import CustomBPE
from src.data.clean import clean_and_dedup_stream
from src.data.config import load_data_config
from src.data.pack import build_bins
from src.data.split import iter_jsonl_texts, resolve_split_files, write_splits_streaming
from src.data.tokenizer_base import compute_file_sha256
from src.data.vocab_eval import (
    analyze_token_usage,
    embedding_param_cost,
    inspect_tokenizer_vocab,
    run_vocab_benchmark,
)

cfg = load_data_config(REPO_ROOT / "configs" / "data.yaml")

if MODE == "colab":
    DRIVE_BASE = Path("/content/drive/MyDrive/minigpt_data")
    INTERIM_DIR = DRIVE_BASE / "interim" / "cleaned_splits"
    SPLIT_STATS_PATH = DRIVE_BASE / "interim" / "split_stats.json"
    BENCHMARK_DIR = DRIVE_BASE / "vocab_benchmark"
    PROCESSED_DIR = DRIVE_BASE / "processed" / "v1_bpe8192"
    TOKENIZER_PATH = PROCESSED_DIR / "tokenizer.json"
    SURVEY_VOCAB_SIZES = [4096, 8192, 16384]
    TRAIN_SAMPLE_MB = 40.0
    TARGET_VOCAB_SIZE = 8192
elif MODE == "fixtures":
    WORK_DIR = REPO_ROOT / "data" / "interim" / "nb_fixtures_work"
    INTERIM_DIR = WORK_DIR / "interim"
    SPLIT_STATS_PATH = INTERIM_DIR / "split_stats.json"
    BENCHMARK_DIR = WORK_DIR / "vocab_benchmark"
    PROCESSED_DIR = WORK_DIR / "processed" / "v1_bpe512"
    TOKENIZER_PATH = PROCESSED_DIR / "tokenizer.json"
    SURVEY_VOCAB_SIZES = [300, 350, 512]
    TRAIN_SAMPLE_MB = 0.5
    TARGET_VOCAB_SIZE = 512
else:
    raise ValueError(f"Unsupported MODE={MODE!r}; expected 'fixtures' or 'colab'")

print(f"MODE={MODE} | REPO_ROOT={REPO_ROOT}")
print(f"INTERIM_DIR={INTERIM_DIR}")
print(f"SPLIT_STATS_PATH={SPLIT_STATS_PATH}")
print(f"PROCESSED_DIR={PROCESSED_DIR}")

In [ ]:
# Nếu chạy ở MODE="fixtures", tạo nhanh split sạch, benchmark mẫu và gói .bin để notebook tự chạy độc lập
if MODE == "fixtures":
    clean_cfg = cfg["cleaning"]
    split_cfg = cfg["split"]
    raw_iter = iter_jsonl_texts(REPO_ROOT / cfg["paths"]["fixtures_file"])
    cleaned_stream = clean_and_dedup_stream(
        raw_iter,
        min_chars=int(clean_cfg["min_chars"]),
        min_valid_char_ratio=float(clean_cfg["min_valid_char_ratio"]),
        min_letter_ratio=float(clean_cfg.get("min_letter_ratio", 0.45)),
        strip_sections=clean_cfg.get("strip_sections"),
    )
    write_splits_streaming(
        cleaned_stream,
        output_dir=INTERIM_DIR,
        ratios=split_cfg["ratios"],
        seed=int(split_cfg["seed"]),
        compress=False,
    )

    split_files_init = resolve_split_files(INTERIM_DIR)
    bench_report = run_vocab_benchmark(
        train_docs_iter=iter_jsonl_texts(split_files_init["train"]),
        val_docs_iter=iter_jsonl_texts(split_files_init["val"]),
        out_dir=BENCHMARK_DIR,
        train_mb=TRAIN_SAMPLE_MB,
        vocab_sizes=SURVEY_VOCAB_SIZES,
        d_models=[256, 384, 512],
        include_hf=True,
    )

    tok_src = BENCHMARK_DIR / f"custom_bpe{TARGET_VOCAB_SIZE}.json"
    bpe_init = CustomBPE.load(tok_src)
    build_bins(
        splits={k: iter_jsonl_texts(split_files_init[k]) for k in ("train", "val", "test")},
        tokenizer=bpe_init,
        output_dir=PROCESSED_DIR,
        version=f"v1_bpe{TARGET_VOCAB_SIZE}",
        tokenizer_type="custom_regex_bpe",
        seed=int(split_cfg["seed"]),
        tokenizer_source_path=tok_src,
        workers=1,
    )
    print("Prepared local fixtures artifacts for notebook execution.")

## (a) Thống kê phân phối độ dài corpus sau làm sạch & khử trùng lặp (`split_stats.json`)

Kiểm tra số lượng tài liệu, tổng số ký tự (tính sau `normalize_text`) và các phân vị độ dài (`min`, `median`, `p95`, `max`, `mean`) trên từng tập `train / val / test`.

In [ ]:
split_stats_path = SPLIT_STATS_PATH
split_stats_payload = json.loads(split_stats_path.read_text(encoding="utf-8"))

stats_df = pd.DataFrame.from_dict(split_stats_payload["stats"], orient="index")
stats_df.index.name = "split"
print(f"Seed: {split_stats_payload['seed']} | Ratios: {split_stats_payload['ratios']}")
stats_df

## (b) Kiểm thử round-trip tiếng Việt & ca biên trên `CustomBPE`

Xác minh `decode(encode(x)) == x` trên các chuỗi ca biên quan trọng và đảm bảo chuỗi literal `<|endoftext|>` không làm phát sinh nhầm `eos_id` khi `allow_special=False`.

In [ ]:
bpe = CustomBPE.load(TOKENIZER_PATH)
print(f"Loaded CustomBPE: vocab_size={bpe.vocab_size}, eos_id={bpe.eos_id}")

edge_cases = {
    "empty": "",
    "vi_nfc": "Hà Nội là thủ đô của Việt Nam; tiếng Việt chuẩn NFC: ă â đ ê ô ơ ư Ắ Ệ Ử.",
    "vi_nfd": unicodedata.normalize("NFD", "Thành phố Hồ Chí Minh và đồng bằng sông Cửu Long."),
    "emoji_and_symbols": "Mô hình ngôn ngữ nhỏ 🚀🔥🇻🇳 — số liệu: 99,5% (°C).",
    "control_chars": "Dòng một\tCột hai\n\nĐoạn hai với ký tự \x00.",
    "literal_eos": "Văn bản chứa chuỗi <|endoftext|> xuất hiện tự nhiên.",
}

rt_rows = []
for name, text in edge_cases.items():
    ids = bpe.encode(text, allow_special=False)
    decoded = bpe.decode(ids)
    assert decoded == text, f"Round-trip failed on {name}"
    assert bpe.eos_id not in ids, f"Unexpected eos_id emitted on {name}"
    rt_rows.append(
        {
            "case": name,
            "utf8_bytes": len(text.encode("utf-8")),
            "bpe_tokens": len(ids),
            "round_trip_ok": decoded == text,
        }
    )

# Kiểm tra thêm allow_special=True
sp_ids = bpe.encode("A<|endoftext|>B", allow_special=True)
assert bpe.eos_id in sp_ids and bpe.decode(sp_ids) == "A<|endoftext|>B"

pd.DataFrame(rt_rows)

## (c) Bảng so sánh các cỡ vocab & Đối chiếu với HuggingFace `tokenizers`

Đọc kết quả benchmark từ `vocab_benchmark.json` (hoặc chạy trực tiếp nếu chưa có) để so sánh `Bytes / Token`, `Chars / Token`, `Fertility (tokens / word)` và tốc độ `Train / Val Encode`.

In [ ]:
bench_json_path = BENCHMARK_DIR / "vocab_benchmark.json"
if bench_json_path.exists():
    bench_data = json.loads(bench_json_path.read_text(encoding="utf-8"))
else:
    split_files_bench = resolve_split_files(INTERIM_DIR)
    bench_data = run_vocab_benchmark(
        train_docs_iter=iter_jsonl_texts(split_files_bench["train"]),
        val_docs_iter=iter_jsonl_texts(split_files_bench["val"]),
        out_dir=BENCHMARK_DIR,
        train_mb=TRAIN_SAMPLE_MB,
        vocab_sizes=SURVEY_VOCAB_SIZES,
        d_models=[256, 384, 512],
        include_hf=True,
    )

comp_cols = [
    "implementation",
    "actual_vocab_size",
    "train_time_sec",
    "val_encode_time_sec",
    "n_tokens",
    "bytes_per_token",
    "chars_per_token",
    "fertility_tokens_per_word",
]
comp_df = pd.DataFrame(bench_data["comparisons"])[comp_cols]
comp_df

## (d) Chi phí tham số Embedding (Weight Tying) & Lý do chốt `vocab_size = 8192`

Với weight tying giữa ma trận Input Embedding và LM Head, tổng tham số lớp từ vựng là $V \times d_{\text{model}}$.

**Lý do nhóm chốt `vocab_size = 8192` cho bản chính `v1_bpe8192`:**
1. **Cân bằng ngân sách tham số (10M–30M):** Ở `vocab_size = 8192` với $d_{\text{model}} \in \{256, 384, 512\}$, phần embedding chỉ chiếm **2,10M – 4,19M** tham số (**21%–42%** của mô hình 10M và **7%–14%** của mô hình 30M), giữ đủ dung lượng cho các khối Transformer decoder. Ngược lại, `16384` tiêu tốn tới **8,39M** tham số ở $d_{\text{model}} = 512$ (gần như chiếm trọn ngân sách 10M).
2. **Độ nén & Fertility hợp lý cho tiếng Việt:** Tách tiếng Việt ở mức subword/âm tiết gọn hơn rõ rệt so với `4096` (giảm chiều dài chuỗi trên cùng `block_size`, giúp ngữ cảnh chứa được nhiều câu hơn), trong khi chi phí softmax đầu ra giữ ở mức nhẹ trên GPU Colab.

In [ ]:
emb_rows = []
for v in [4096, 8192, 16384]:
    for item in embedding_param_cost(vocab_size=v, d_models=[256, 384, 512]):
        emb_rows.append({"vocab_size": v, **item})

emb_df = pd.DataFrame(emb_rows)
emb_df

## (e) Kiểm định chất lượng nội bộ vocab & Phân tích token sử dụng trên `.bin`

Sử dụng `inspect_tokenizer_vocab` để kiểm tra tỷ lệ token chữ cái tiếng Việt/Latinh hợp lệ, token khoảng trắng thuần, token byte cắt dở và các token ghép tiêu biểu; sau đó dùng `analyze_token_usage` (`np.bincount`) trên `train.bin` (hoặc `val.bin`/`test.bin`) để phát hiện token chết (tần suất = 0).

In [ ]:
vocab_quality = inspect_tokenizer_vocab(bpe, sample_limit=25)
print("=== Tóm tắt chất lượng từ vựng đã học ===")
summary_quality = {
    k: v
    for k, v in vocab_quality.items()
    if k not in ("first_learned_tokens", "longest_letter_tokens")
}
print(json.dumps(summary_quality, ensure_ascii=False, indent=2))

print("\n--- 15 token ghép đầu tiên ---")
display_first = pd.DataFrame(vocab_quality["first_learned_tokens"][:15])
print(display_first.to_string(index=False))

print("\n--- 15 token chữ cái dài nhất ---")
display_longest = pd.DataFrame(vocab_quality["longest_letter_tokens"][:15])
print(display_longest.to_string(index=False))

usage_train = analyze_token_usage(
    PROCESSED_DIR / "train.bin",
    vocab_size=bpe.vocab_size,
    tokenizer=bpe,
    top_k=15,
)
print(
    f"\n=== Tần suất token trên train.bin ===\n"
    f"Total tokens: {usage_train['total_tokens']:,} | "
    f"Dead tokens (0..V-1): {usage_train['dead_token_count']} ({usage_train['dead_token_ratio']*100:.2f}%) | "
    f"Learned dead tokens (257..V-1): {usage_train['learned_dead_token_count']}"
)
pd.DataFrame(usage_train["top_tokens"])

## (f) Kiểm tra hợp đồng `meta.json` và giải mã mẫu từ `test.bin`

Đối chiếu `tokenizer_hash`, đếm số `eos_id` bằng `n_docs - 1`, xác minh toàn bộ ID `< vocab_size` và đọc ngược một đoạn thực tế quanh ranh giới `<|endoftext|>` trong `test.bin`.

In [ ]:
meta = json.loads((PROCESSED_DIR / "meta.json").read_text(encoding="utf-8"))
actual_hash = compute_file_sha256(TOKENIZER_PATH)
assert meta["tokenizer_hash"] == actual_hash, "tokenizer_hash mismatch!"
assert meta["vocab_size"] == bpe.vocab_size
assert meta["eos_id"] == bpe.eos_id

print(json.dumps(meta, ensure_ascii=False, indent=2))

for split_name in ("train", "val", "test"):
    arr = np.memmap(PROCESSED_DIR / f"{split_name}.bin", dtype="<u2", mode="r")
    s_meta = meta["splits"][split_name]
    assert len(arr) == s_meta["n_tokens"]
    if len(arr) > 0:
        assert int(arr.max()) < meta["vocab_size"]
        eos_count = int(np.sum(arr == meta["eos_id"]))
        assert eos_count == max(0, s_meta["n_docs"] - 1)
        print(f"[{split_name}] OK: n_tokens={len(arr):,}, n_docs={s_meta['n_docs']:,}, eos_count={eos_count:,}")

# Giải mã một đoạn quanh ranh giới eos_id đầu tiên trong test.bin
test_arr = np.memmap(PROCESSED_DIR / "test.bin", dtype="<u2", mode="r")
eos_positions = np.where(test_arr == meta["eos_id"])[0]
if len(eos_positions) > 0:
    pos = int(eos_positions[0])
    lo = max(0, pos - 60)
    hi = min(len(test_arr), pos + 80)
    decoded_window = bpe.decode(test_arr[lo:hi].tolist())
    print("\n--- Mẫu giải mã quanh ranh giới tài liệu đầu tiên trong test.bin ---")
    print(decoded_window)
else:
    print("\n--- Mẫu giải mã test.bin ---")
    print(bpe.decode(test_arr[:120].tolist()))